<a href="https://colab.research.google.com/github/CuriousTechNomad/slm-agentic-software-engineering/blob/main/notebooks/03_baseline_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 - Baseline Evaluation

## Objective

Evaluate the baseline Small Language Model (SLM) on the official SWE-Bench Lite evaluation set.

This notebook:

- Loads the official evaluation tasks
- Loads the baseline SLM
- Generates a solution for every task
- Measures inference statistics
- Saves results for later comparison

These results become the baseline for:

- Multi-Agent workflow
- Fine-tuned SLM

In [2]:
# ======================================================
# Project Setup
# ======================================================

from google.colab import drive
from pathlib import Path
import os
import json

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/LLM_Project")

OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR = PROJECT_DIR / "data"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
CONFIG_DIR = PROJECT_DIR / "configs"

for folder in [
    OUTPUT_DIR,
    DATA_DIR,
    CHECKPOINT_DIR,
    CONFIG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project initialized")
print(PROJECT_DIR)

Mounted at /content/drive
Project initialized
/content/drive/MyDrive/LLM_Project


In [3]:
!pip -q install transformers accelerate pandas

In [4]:
import json
import time
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [5]:
config = {
    "model_name": "Qwen/Qwen2.5-3B-Instruct",
    "development_tasks": 3,
    "max_new_tokens": 300,
    "temperature": 0.2
}

print(config)

{'model_name': 'Qwen/Qwen2.5-3B-Instruct', 'development_tasks': 3, 'max_new_tokens': 300, 'temperature': 0.2}


In [6]:
evaluation_file = OUTPUT_DIR / "official_evaluation_set.json"

with open(evaluation_file) as f:
    tasks = json.load(f)

print(f"Loaded {len(tasks)} evaluation tasks")

Loaded 20 evaluation tasks


In [7]:
NUM_TASKS = config["development_tasks"]

tasks = tasks[:NUM_TASKS]

print(f"Running {len(tasks)} tasks")

Running 3 tasks


In [8]:
MODEL_NAME = config["model_name"]

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model Ready")

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model Ready


In [9]:
PROMPT_TEMPLATE = """
You are a senior software engineer.

Repository:
{repo}

GitHub Issue:
{issue}

Analyze the issue carefully.

Explain:

1. Root cause

2. Files likely involved

3. High-level implementation plan

Do not generate a git patch.

Keep the answer technical.
"""

In [18]:
def run_inference(task):

    prompt = PROMPT_TEMPLATE.format(
        repo=task["repo"],
        issue=task["problem_statement"]
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    start = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=config["max_new_tokens"],
        temperature=config["temperature"],
        do_sample=False
    )

    elapsed = time.time() - start

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    # Return all information for this task
    return {
        "instance_id": task["instance_id"],
        "repo": task["repo"],
        "problem_statement": task["problem_statement"],
        "base_commit": task["base_commit"],
        "inference_time": round(elapsed, 2),
        "input_tokens": inputs.input_ids.shape[1],
        "output_tokens": outputs.shape[1] - inputs.input_ids.shape[1],
        "response": response
    }

In [19]:
results = []

for i, task in enumerate(tasks):

    print("=" * 80)
    print(f"{i+1}/{len(tasks)}")
    print(task["instance_id"])

    result = run_inference(task)

    results.append(result)

    print(f"Completed in {result['inference_time']} sec")

    # Save progress after every task
    pd.DataFrame(results).to_csv(
        OUTPUT_DIR / "baseline_results_partial.csv",
        index=False
    )

1/3
pallets__flask-4045
Completed in 19.4 sec
2/3
pytest-dev__pytest-11143
Completed in 18.85 sec
3/3
astropy__astropy-12907
Completed in 16.47 sec


In [20]:
baseline_df = pd.DataFrame(results)

baseline_df.head()

,instance_id,repo,problem_statement,base_commit,inference_time,input_tokens,output_tokens,response
0,pallets__flask-4045,pallets/flask,Raise error when blueprint name contains a dot...,d8c37f43724cd9fb0870f77877b7c4c7e38a19e0,19.40,107,300,### Analysis\n\n#### 1. Root Cause\n\nThe root...
1,pytest-dev__pytest-11143,pytest-dev/pytest,Rewrite fails when first expression of file is...,6995257cf470d2143ad1683824962de4071c0eb7,18.85,1888,300,### Analysis\n\n#### Root Cause\n\nThe error o...
2,astropy__astropy-12907,astropy/astropy,Modeling's `separability_matrix` does not comp...,d16bfe05a744909de4b27f5875fe0d4ed41ce607,16.47,382,300,### Analysis\n\n#### 1. Root Cause\n\nThe root...


In [21]:
baseline_df.to_csv(
    OUTPUT_DIR / "baseline_results.csv",
    index=False
)

baseline_df.to_json(
    OUTPUT_DIR / "baseline_results.json",
    orient="records",
    indent=2
)

print("Results saved successfully")

Results saved successfully


In [22]:
print("Average Inference Time")

print(
    baseline_df["inference_time"].mean()
)

print()

print("Average Input Tokens")

print(
    baseline_df["input_tokens"].mean()
)

print()

print("Average Output Tokens")

print(
    baseline_df["output_tokens"].mean()
)

Average Inference Time
18.24

Average Input Tokens
792.3333333333334

Average Output Tokens
300.0


In [24]:
print("="*80)

print("Repository")

print(baseline_df.iloc[0]["repo"])

print()

print("Response")

print(baseline_df.iloc[0]["response"])

Repository
pallets/flask

Response
### Analysis

#### 1. Root Cause

The root cause of this issue lies in the current design and implementation of Flask's blueprint system. Blueprints in Flask allow for modular and reusable components within an application. They can be nested, which means that a blueprint can be defined within another blueprint. This nesting introduces a level of complexity where the structure of the blueprint names becomes significant. The requirement to raise an error when a blueprint name contains a dot stems from the need to ensure that blueprint names remain unique and unambiguous, especially when they are nested. If a blueprint name contains a dot, it could lead to ambiguity or conflicts, making it difficult to manage and understand the structure of the application.

#### 2. Files Likely Involved

Several files in the `flask` repository are likely to be involved in addressing this issue:

- **`flask/blueprints.py`**: This file contains the core logic for handling

In [25]:
experiment = {
    "experiment": "baseline",
    "model": MODEL_NAME,
    "tasks": len(tasks),
    "average_inference_time": baseline_df["inference_time"].mean(),
    "average_input_tokens": baseline_df["input_tokens"].mean(),
    "average_output_tokens": baseline_df["output_tokens"].mean()
}

with open(
    OUTPUT_DIR / "baseline_experiment.json",
    "w"
) as f:

    json.dump(experiment, f, indent=4)

print("Experiment log saved")

Experiment log saved


# Observations

## Strengths

- Produces technical reasoning.
- Understands software engineering concepts.
- Explains implementation strategy.

## Limitations

- No repository context.
- Cannot inspect source code.
- Cannot search files.
- Cannot validate implementation.

## Next Notebook

Notebook 04 will introduce a Multi-Agent workflow where specialized agents retrieve repository context before generating the final solution.